# MuSSED Model Interactive Testing

This notebook provides step-by-step testing of the MuSSED model for event detection.
Run each cell sequentially to test different aspects of the model.

**Stages:**
1. **Setup & Configuration** - Imports and config
2. **Model Building** - Initialize MuSSED
3. **Synthetic Data Tests** - Test with various tensor shapes
4. **Real Data Tests** - Test with actual NVCHVC dataset
5. **Edge Cases** - Stress test the model
6. **Summary** - Aggregate and report results

## Stage 1: Setup & Configuration

In [6]:
import sys
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import torch
from torch.utils.data import DataLoader

# Setup paths - find project root by looking for models directory
current_path = Path.cwd()
project_root = current_path

# Try to find project root by going up directories looking for 'models' folder
for parent in [current_path] + list(current_path.parents):
    if (parent / "models").exists() and (parent / "utils").exists():
        project_root = parent
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Current working directory: {current_path}")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

Project root: c:\CAMILO\Volcanes_UFRO\CODES\graph-volcano
Current working directory: c:\CAMILO\Volcanes_UFRO\CODES\graph-volcano\notebooks
PyTorch version: 2.6.0+cu126
Device available: cuda


In [7]:
# Device and dtype configuration
DEVICE = torch.device("cpu")#"cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

print(f"Using device: {DEVICE}")
print(f"Using dtype: {DTYPE}\n")

# MuSSED Model Configuration (best ablation: musseg_d4_r16_s222_k127)
MUSSED_CONFIG = {
    # Encoder parameters
    "in_channels": 8,
    "num_classes": 6,
    "depth": 4,
    "kernel_size": 127,
    "stride": [2, 2, 2],
    "dilation": [1, 1, 1, 1],
    "filters_root": 16,
    "norm": "std",
    "feature_dropout": 0.0,
    "bottleneck_attention": True,
    "bottleneck_attn_heads": 4,
    "bottleneck_attn_dropout": 0.0,
    "bottleneck_attn_ff_mult": 2,
    "station_attn_heads": 4,
    "station_attn_dropout": 0.0,
    "station_attn_ff_mult": 2,
    "volcano_name": "NVCHVC",
    "use_distance_attn_bias": False,
    "use_distance_bottleneck_emb": False,
    "use_station_weighted_skips": False,
    # Detection head parameters
    "num_queries": 3,
    "query_dim": 128,
    "hidden_dim": 256,
    "num_decoder_heads": 4,
    "num_decoder_layers": 2,
    "decoder_dropout": 0.1,
}

print("MuSSED Configuration:")
print(json.dumps(MUSSED_CONFIG, indent=2))

Using device: cpu
Using dtype: torch.float32

MuSSED Configuration:
{
  "in_channels": 8,
  "num_classes": 6,
  "depth": 4,
  "kernel_size": 127,
  "stride": [
    2,
    2,
    2
  ],
  "dilation": [
    1,
    1,
    1,
    1
  ],
  "filters_root": 16,
  "norm": "std",
  "feature_dropout": 0.0,
  "bottleneck_attention": true,
  "bottleneck_attn_heads": 4,
  "bottleneck_attn_dropout": 0.0,
  "bottleneck_attn_ff_mult": 2,
  "station_attn_heads": 4,
  "station_attn_dropout": 0.0,
  "station_attn_ff_mult": 2,
  "volcano_name": "NVCHVC",
  "use_distance_attn_bias": false,
  "use_distance_bottleneck_emb": false,
  "use_station_weighted_skips": false,
  "num_queries": 3,
  "query_dim": 128,
  "hidden_dim": 256,
  "num_decoder_heads": 4,
  "num_decoder_layers": 2,
  "decoder_dropout": 0.1
}


In [3]:
# Simple utility functions
def format_shape(shape):
    """Format tensor shape for display"""
    return f"[{', '.join(str(s) for s in shape)}]"

def check_tensor_health(tensor, name, expected_shape=None):
    """Quick check of tensor health"""
    print(f"\n{name}:")
    print(f"  Shape: {format_shape(tensor.shape)}", end="")
    if expected_shape and tensor.shape != expected_shape:
        print(f" ✗ Expected {format_shape(expected_shape)}")
        return False
    print(" ✓")
    
    if torch.isnan(tensor).any():
        print(f"  ✗ Contains {int(torch.isnan(tensor).sum())} NaN values!")
        return False
    
    if torch.isinf(tensor).any():
        print(f"  ✗ Contains {int(torch.isinf(tensor).sum())} Inf values!")
        return False
    
    val_min = float(tensor.min())
    val_max = float(tensor.max())
    val_mean = float(tensor.mean())
    print(f"  Range: [{val_min:.4f}, {val_max:.4f}], Mean: {val_mean:.4f}")
    return True

# Testing results storage
test_results = []

print("✓ Utility functions defined")

✓ Utility functions defined


## Stage 2: Model Building & Initialization

In [8]:
from models.MuSSED import MuSSED

print("Building MuSSED model...")
model = MuSSED(**MUSSED_CONFIG)
model = model.to(DEVICE)
model.eval()

print("✓ Model built successfully")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Parameters:")
print(f"  Total:     {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

# Show model structure summary
print(f"\nModel Structure:")
print(f"  Encoder: {model.encoder.__class__.__name__}")
print(f"  Decoder: {model.decoder.__class__.__name__}")
print(f"  Detection Head: {model.detection_head.__class__.__name__}")
print(f"  Queries: {model.num_queries} (shape: [1, {model.num_queries}, {model.query_dim}])")
print(f"  Output channels: {model.encoder.output_channels}")

Building MuSSED model...
✓ Model built successfully

Model Parameters:
  Total:     5,678,913
  Trainable: 5,678,913

Model Structure:
  Encoder: TemporalEncoder
  Decoder: DETRTransformerDecoder
  Detection Head: DetectionHead
  Queries: 3 (shape: [1, 3, 128])
  Output channels: 128


## Stage 3: Synthetic Data Tests

Testing with randomly generated tensors of various shapes

In [9]:
# Define synthetic test configurations
# (batch_size, num_stations, time_length, description)
synthetic_tests = [
    (1, 1, 2048, "Single station, short"),
    (1, 5, 4096, "Multi-station, medium"),
    (2, 3, 8192, "Small batch, multi-station"),
    (4, 5, 8192, "Medium batch, 5 stations"),
    (8, 2, 4096, "Larger batch, 2 stations"),
    (4, 1, 16384, "Single station, long"),
    (2, 10, 6000, "Many stations, medium"),
    (1, 7, 12288, "Multi-station, long"),
]

print(f"Configured {len(synthetic_tests)} synthetic test cases")
for batch, stations, time, desc in synthetic_tests:
    print(f"  • {format_shape((batch, stations, time))} - {desc}")

Configured 8 synthetic test cases
  • [1, 1, 2048] - Single station, short
  • [1, 5, 4096] - Multi-station, medium
  • [2, 3, 8192] - Small batch, multi-station
  • [4, 5, 8192] - Medium batch, 5 stations
  • [8, 2, 4096] - Larger batch, 2 stations
  • [4, 1, 16384] - Single station, long
  • [2, 10, 6000] - Many stations, medium
  • [1, 7, 12288] - Multi-station, long


In [10]:
# Run FIRST synthetic test as example
print("="*70)
print("SYNTHETIC TEST #1: Single station, short")
print("="*70)

batch_size, num_stations, time_length = 1, 1, 2048
x_test1 = torch.randn(batch_size, num_stations, time_length, dtype=DTYPE, device=DEVICE)

print(f"\nInput shape: {format_shape(x_test1.shape)}")
print(f"  Batch size: {batch_size}")
print(f"  Stations: {num_stations}")
print(f"  Time samples: {time_length}")

# Forward pass
with torch.no_grad():
    output = model(x_test1)

print(f"\n✓ Forward pass completed!")

# Validate outputs
print(f"\nOutput validation:")
all_ok = True
all_ok &= check_tensor_health(output["class_logits"], "class_logits", 
                              (batch_size, MUSSED_CONFIG["num_queries"], MUSSED_CONFIG["num_classes"]))
all_ok &= check_tensor_health(output["center"], "center", 
                              (batch_size, MUSSED_CONFIG["num_queries"], 1))
all_ok &= check_tensor_health(output["start"], "start", 
                              (batch_size, MUSSED_CONFIG["num_queries"], 1))
all_ok &= check_tensor_health(output["end"], "end", 
                              (batch_size, MUSSED_CONFIG["num_queries"], 1))
all_ok &= check_tensor_health(output["confidence"], "confidence", 
                              (batch_size, MUSSED_CONFIG["num_queries"], 1))
all_ok &= check_tensor_health(output["encoder_features"], "encoder_features")

print(f"\n{'✓ Test 1 PASSED' if all_ok else '✗ Test 1 FAILED'}")
test_results.append(("Test 1: Single station, short", all_ok))

SYNTHETIC TEST #1: Single station, short

Input shape: [1, 1, 2048]
  Batch size: 1
  Stations: 1
  Time samples: 2048

✓ Forward pass completed!

Output validation:

class_logits:
  Shape: [1, 3, 6] ✓
  Range: [-0.3040, 0.4919], Mean: 0.0944

center:
  Shape: [1, 3, 1] ✓
  Range: [-0.2797, -0.1855], Mean: -0.2320

start:
  Shape: [1, 3, 1] ✓
  Range: [0.1617, 0.5352], Mean: 0.3756

end:
  Shape: [1, 3, 1] ✓
  Range: [0.0414, 0.2061], Mean: 0.1148

confidence:
  Shape: [1, 3, 1] ✓
  Range: [-0.2536, 0.4431], Mean: 0.0697

encoder_features:
  Shape: [1, 128, 256] ✓
  Range: [-1.4915, 1.6391], Mean: 0.0117

✓ Test 1 PASSED


In [11]:
# Run remaining synthetic tests (Test 2-8)
for test_idx, (batch_size, num_stations, time_length, description) in enumerate(synthetic_tests[1:], start=2):
    print(f"\n{'='*70}")
    print(f"SYNTHETIC TEST #{test_idx}: {description}")
    print(f"{'='*70}")
    
    x = torch.randn(batch_size, num_stations, time_length, dtype=DTYPE, device=DEVICE)
    print(f"Input: {format_shape(x.shape)} - ", end="")
    
    try:
        with torch.no_grad():
            output = model(x)
        
        # Quick validation
        all_ok = True
        all_ok &= output["class_logits"].shape == (batch_size, MUSSED_CONFIG["num_queries"], MUSSED_CONFIG["num_classes"])
        all_ok &= output["center"].shape == (batch_size, MUSSED_CONFIG["num_queries"], 1)
        all_ok &= not torch.isnan(output["class_logits"]).any()
        all_ok &= not torch.isinf(output["class_logits"]).any()
        
        status = "✓ PASSED" if all_ok else "✗ FAILED"
        print(status)
        test_results.append((f"Test {test_idx}: {description}", all_ok))
        
    except Exception as e:
        print(f"✗ FAILED - {type(e).__name__}: {str(e)[:50]}")
        test_results.append((f"Test {test_idx}: {description}", False))


SYNTHETIC TEST #2: Multi-station, medium
Input: [1, 5, 4096] - ✓ PASSED

SYNTHETIC TEST #3: Small batch, multi-station
Input: [2, 3, 8192] - ✓ PASSED

SYNTHETIC TEST #4: Medium batch, 5 stations
Input: [4, 5, 8192] - ✓ PASSED

SYNTHETIC TEST #5: Larger batch, 2 stations
Input: [8, 2, 4096] - ✓ PASSED

SYNTHETIC TEST #6: Single station, long
Input: [4, 1, 16384] - ✓ PASSED

SYNTHETIC TEST #7: Many stations, medium
Input: [2, 10, 6000] - ✓ PASSED

SYNTHETIC TEST #8: Multi-station, long
Input: [1, 7, 12288] - ✓ PASSED


## Stage 4: Real Data Tests

Loading and testing with actual NVCHVC dataset

In [17]:
from utils.musseg_utils import MuSSegWindowDataset, musseg_collate_fn

# Locate data - use project root instead of cwd
data_root = project_root / "data" / "prepared_data"
fold_root = data_root / "NVCHVC" / "cv_5fold" / "fold_01"

print(f"Project root: {project_root}\n")
print(f"Looking for fold_root: {fold_root}")
print(f"fold_root exists: {fold_root.exists()}\n")

# First, check what's ACTUALLY in the directory
if fold_root.exists():
    print(f"Contents of {fold_root}:")
    all_items = list(fold_root.iterdir())
    for item in sorted(all_items):
        size_str = f" ({item.stat().st_size / 1024:.1f} KB)" if item.is_file() else ""
        print(f"  {'📄' if item.is_file() else '📁'} {item.name}{size_str}")
    print()
    
    # Find ALL npz files
    npz_files = list(fold_root.glob("*.npz"))
    if npz_files:
        print(f"Found {len(npz_files)} npz file(s):")
        for f in sorted(npz_files):
            size_mb = f.stat().st_size / (1024**2)
            print(f"  ✓ {f.name} ({size_mb:.1f} MB)")
        
        # Use first one found
        train_manifest = sorted(npz_files)[0]
        print(f"\nUsing: {train_manifest.name}\n")
    else:
        print("⚠ NO NPZ FILES FOUND!")
        print("Checked directory contents above ☝️\n")
        raise FileNotFoundError(f"No .npz files in {fold_root}")
else:
    raise FileNotFoundError(f"Fold root directory not found: {fold_root}")

print("Loading NVCHVC dataset...")
dataset = MuSSegWindowDataset(
    train_manifest,
    num_classes=6,
    station_rows=None,  # Infer from data
    use_zero_mask=True,
    scramble_stations=False,
)

print(f"✓ Dataset loaded from {train_manifest.name}")
print(f"  Samples: {len(dataset)}")

# Create DataLoader
loader = DataLoader(
    dataset,
    batch_size=2,
    collate_fn=musseg_collate_fn,
    shuffle=False,
    num_workers=0,
)

print(f"  Batches: {len(loader)}")

Project root: c:\CAMILO\Volcanes_UFRO\CODES\graph-volcano

Looking for fold_root: c:\CAMILO\Volcanes_UFRO\CODES\graph-volcano\data\prepared_data\NVCHVC\cv_5fold\fold_01
fold_root exists: True

Contents of c:\CAMILO\Volcanes_UFRO\CODES\graph-volcano\data\prepared_data\NVCHVC\cv_5fold\fold_01:
  📁 augmented
  📁 edge_data
  📄 test.npz (578.3 KB)
  📄 train.npz (1959.7 KB)
  📄 train_aug.npz (4611.3 KB)
  📄 val.npz (347.3 KB)

Found 4 npz file(s):
  ✓ test.npz (0.6 MB)
  ✓ train.npz (1.9 MB)
  ✓ train_aug.npz (4.5 MB)
  ✓ val.npz (0.3 MB)

Using: test.npz

Loading NVCHVC dataset...
✓ Dataset loaded from test.npz
  Samples: 1822
  Batches: 911


In [19]:
# Test with real data (first 2 batches)
print("="*70)
print("REAL DATA TESTS")
print("="*70)

for batch_idx, batch in enumerate(loader):
    if batch_idx >= 2:  # Only first 2 batches
        break
    
    print(f"\n{'-'*70}")
    print(f"Real Data Batch {batch_idx + 1}")
    print(f"{'-'*70}")
    
    x = batch.x.to(DEVICE).float()
    print(f"Input shape: {format_shape(x.shape)}")
    print(f"  Batch: {x.shape[0]}, Stations: {x.shape[1]}, Time: {x.shape[2]}")
    
    try:
        with torch.no_grad():
            output = model(x)
        
        # Validate
        all_ok = True
        all_ok &= output["class_logits"].shape[0] == x.shape[0]
        all_ok &= output["class_logits"].shape[1] == MUSSED_CONFIG["num_queries"]
        all_ok &= not torch.isnan(output["class_logits"]).any()
        
        print(f"\nOutput shapes:")
        print(f"  class_logits: {format_shape(output['class_logits'].shape)}")
        print(f"  center: {format_shape(output['center'].shape)}")
        print(f"  start: {format_shape(output['start'].shape)}")
        print(f"  end: {format_shape(output['end'].shape)}")
        print(f"  confidence: {format_shape(output['confidence'].shape)}")
        print(f"  encoder_features: {format_shape(output['encoder_features'].shape)}")
        
        print(f"\nOutput values (sample from batch):")
        # Show first sample, first query
        print(f"\n  Query 1 (first sample):")
        print(f"    Class logits:  {output['class_logits'][0, 0].cpu().numpy().round(3)}")
        print(f"    Class (argmax): {int(output['class_logits'][0, 0].argmax())}")
        print(f"    Center time:   {float(output['center'][0, 0, 0]):.2f}")
        print(f"    Start time:    {float(output['start'][0, 0, 0]):.2f}")
        print(f"    End time:      {float(output['end'][0, 0, 0]):.2f}")
        print(f"    Confidence:    {float(output['confidence'][0, 0, 0]):.4f}")
        
        if MUSSED_CONFIG["num_queries"] > 1:
            print(f"\n  Query 2 (first sample):")
            print(f"    Class logits:  {output['class_logits'][0, 1].cpu().numpy().round(3)}")
            print(f"    Class (argmax): {int(output['class_logits'][0, 1].argmax())}")
            print(f"    Center time:   {float(output['center'][0, 1, 0]):.2f}")
            print(f"    Start time:    {float(output['start'][0, 1, 0]):.2f}")
            print(f"    End time:      {float(output['end'][0, 1, 0]):.2f}")
            print(f"    Confidence:    {float(output['confidence'][0, 1, 0]):.4f}")
        
        status = "✓ PASSED" if all_ok else "✗ FAILED"
        print(f"\n{status}")
        test_results.append((f"Real data batch {batch_idx + 1}", all_ok))
        
    except Exception as e:
        print(f"✗ FAILED - {type(e).__name__}: {e}")
        test_results.append((f"Real data batch {batch_idx + 1}", False))

REAL DATA TESTS

----------------------------------------------------------------------
Real Data Batch 1
----------------------------------------------------------------------
Input shape: [2, 8, 8192]
  Batch: 2, Stations: 8, Time: 8192

Output shapes:
  class_logits: [2, 3, 6]
  center: [2, 3, 1]
  start: [2, 3, 1]
  end: [2, 3, 1]
  confidence: [2, 3, 1]
  encoder_features: [2, 128, 1024]

Output values (sample from batch):

  Query 1 (first sample):
    Class logits:  [ 0.251  0.019  0.508 -0.082  0.315 -0.204]
    Class (argmax): 2
    Center time:   -0.32
    Start time:    0.36
    End time:      0.23
    Confidence:    0.0087

  Query 2 (first sample):
    Class logits:  [ 0.18   0.035 -0.179  0.136  0.326 -0.067]
    Class (argmax): 4
    Center time:   -0.25
    Start time:    0.09
    End time:      0.08
    Confidence:    -0.2224

✓ PASSED

----------------------------------------------------------------------
Real Data Batch 2
---------------------------------------------

## Stage 5: Edge Case Tests

Stress testing with extreme input configurations

In [ ]:
# Define edge cases
edge_cases = [
    (1, 1, 512, "Very short sequence"),
    (1, 1, 32768, "Very long sequence"),
    (1, 20, 2048, "Many stations"),
    (16, 5, 2048, "Large batch"),
]

print(f"Configured {len(edge_cases)} edge case tests:")
for batch, stations, time, desc in edge_cases:
    print(f"  • {format_shape((batch, stations, time))} - {desc}")

In [ ]:
print("\n" + "="*70)
print("EDGE CASE TESTS")
print("="*70)

for edge_idx, (batch_size, num_stations, time_length, description) in enumerate(edge_cases, start=1):
    print(f"\n{'-'*70}")
    print(f"Edge Case {edge_idx}: {description}")
    print(f"{'-'*70}")
    
    x = torch.randn(batch_size, num_stations, time_length, dtype=DTYPE, device=DEVICE)
    print(f"Input: {format_shape(x.shape)} - ", end="")
    
    try:
        with torch.no_grad():
            output = model(x)
        
        # Validate
        all_ok = True
        all_ok &= output["class_logits"].shape[0] == batch_size
        all_ok &= not torch.isnan(output["class_logits"]).any()
        all_ok &= not torch.isinf(output["class_logits"]).any()
        
        status = "✓ PASSED" if all_ok else "✗ FAILED"
        print(status)
        print(f"  Output shape: {format_shape(output['class_logits'].shape)}")
        test_results.append((f"Edge case {edge_idx}: {description}", all_ok))
        
    except Exception as e:
        print(f"✗ FAILED")
        print(f"  Error: {type(e).__name__}: {str(e)[:60]}")
        test_results.append((f"Edge case {edge_idx}: {description}", False))

## Stage 6: Test Summary & Report

In [ ]:
print("\n" + "="*70)
print("TEST SUMMARY")
print("="*70)

# Aggregate results
total_tests = len(test_results)
passed_tests = sum(1 for _, result in test_results if result)
failed_tests = total_tests - passed_tests

print(f"\nTotal Tests:  {total_tests}")
print(f"Passed:       {passed_tests}")
print(f"Failed:       {failed_tests}")
print(f"Success Rate: {100 * passed_tests / total_tests if total_tests > 0 else 0:.1f}%")

# Detailed results
print(f"\n{'-'*70}")
print("Detailed Results:")
print(f"{'-'*70}")

for test_name, result in test_results:
    status = "✓ PASSED" if result else "✗ FAILED"
    print(f"{status:10} - {test_name}")

# Summary verdict
print(f"\n{'='*70}")
if failed_tests == 0:
    print("✓ ALL TESTS PASSED!")
else:
    print(f"✗ {failed_tests} test(s) failed. Review errors above.")
print(f"{'='*70}")

In [ ]:
# Optional: Save test report to JSON
report = {
    "timestamp": datetime.now().isoformat(),
    "device": str(DEVICE),
    "model_config": MUSSED_CONFIG,
    "summary": {
        "total_tests": total_tests,
        "passed": passed_tests,
        "failed": failed_tests,
        "success_rate": 100 * passed_tests / total_tests if total_tests > 0 else 0,
    },
    "test_results": [
        {"name": name, "passed": result}
        for name, result in test_results
    ]
}

print("\n" + "="*70)
print("SAVING REPORT")
print("="*70)

report_path = project_root / "results" / "test_report_mussed.json"
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"\n✓ Report saved to: {report_path}")
print(f"\nReport preview:")
print(json.dumps(report, indent=2))

## Bonus: Gradient Flow Test (Optional)

Verify that gradients flow correctly through the model for training

In [ ]:
print("\n" + "="*70)
print("GRADIENT FLOW TEST")
print("="*70)

# Create small test input
x_grad_test = torch.randn(2, 3, 4096, dtype=DTYPE, device=DEVICE, requires_grad=False)

print(f"\nTest input: {format_shape(x_grad_test.shape)}")

# Forward pass
outputs = model(x_grad_test)

# Create dummy loss (sum of predictions)
loss = outputs["class_logits"].sum() + outputs["confidence"].sum()

print(f"Loss value: {float(loss):.6f}")

# Backward pass
print("Running backward pass...", end=" ")
try:
    loss.backward()
    print("✓")
    
    # Check gradients
    has_gradients = False
    params_with_grad = 0
    
    for name, param in model.named_parameters():
        if param.grad is not None and param.grad.abs().sum() > 0:
            has_gradients = True
            params_with_grad += 1
    
    total_params = sum(1 for _ in model.parameters())
    
    print(f"\n✓ Gradient flow successful!")
    print(f"  Parameters with gradients: {params_with_grad}/{total_params}")
    
    if params_with_grad == total_params:
        print(f"  ✓ All parameters receiving gradients")
    else:
        print(f"  ⚠ Some parameters not receiving gradients")
        
except Exception as e:
    print(f"✗")
    print(f"Error: {type(e).__name__}: {e}")